## Next Steps

Now that you understand fine-tuning, explore other training recipes:

- **03_pretraining.ipynb**: Pre-train models from scratch on raw text data
- **04_alignment.ipynb**: Align models with human preferences using RLHF and DPO
- **05_grpo.ipynb**: Group Relative Policy Optimization for advanced alignment
- **06_advanced.ipynb**: Advanced techniques — custom datasets, callbacks, distributed training

For production deployments, check out the Training Studio documentation for experiment tracking, hyperparameter tuning, and model versioning.

In [ ]:
from trainlib.export import merge

# Merge LoRA adapters into the base model
merge("output/lora-finetune", save_to="output/merged-model")

# The merged model can now be used like any other model
# No need to load adapters separately during inference

## Merging LoRA Weights

After training, LoRA adapters can be merged back into the base model for deployment:

- Creates a standalone model without adapter overhead
- Simplifies inference — no need to load adapters separately
- Slightly faster inference than loading adapters dynamically
- Results in a full-size model checkpoint

You can also keep adapters separate for multi-adapter scenarios or easier sharing.

In [ ]:
from trainlib.config.schema import (
    DataConfig,
    EvalConfig,
    LoggingConfig,
    LoraConfig,
    ModelConfig,
    OutputConfig,
    TrainConfig,
    TrainerConfig,
)

config = TrainConfig(
    recipe="finetune",
    method="lora",
    model=ModelConfig(
        name="meta-llama/Llama-3.1-8B",
        dtype="bfloat16",
    ),
    data=DataConfig(
        path="data/train.jsonl",
        format="alpaca",
        eval_split=0.05,
        max_seq_length=2048,
        packing=True,
    ),
    lora=LoraConfig(rank=16, alpha=32),
    trainer=TrainerConfig(
        batch_size=4,
        gradient_accumulation=4,
        learning_rate=2e-4,
        num_epochs=3,
        mixed_precision="bf16",
        warmup_steps=100,
        checkpoint_every_n_steps=500,
    ),
    eval=EvalConfig(every_n_steps=500, metrics=["loss", "perplexity"]),
    logging=LoggingConfig(backends=["console", "tensorboard"], project="my-finetune"),
    output=OutputConfig(dir="output/lora-finetune"),
)

state = trainlib.finetune(config=config)

## Full Configuration Example

For complete control, use the full TrainConfig schema with all sections:

- **model**: Model architecture, dtype, quantization
- **data**: Dataset paths, formats, preprocessing
- **lora**: LoRA-specific parameters (when using LoRA/QLoRA)
- **trainer**: Training hyperparameters, optimization, checkpointing
- **eval**: Evaluation schedule and metrics
- **logging**: TensorBoard, W&B, console logging
- **output**: Where to save checkpoints and final model

In [ ]:
# Alpaca format: {"instruction": "...", "input": "...", "output": "..."}
state = trainlib.finetune(
    model="meta-llama/Llama-3.1-8B",
    dataset="alpaca_data.jsonl",
    format="alpaca",
    method="lora",
)

# ShareGPT format: {"conversations": [{"from": "human", "value": "..."}, ...]}
state = trainlib.finetune(
    model="meta-llama/Llama-3.1-8B",
    dataset="sharegpt_data.jsonl",
    format="sharegpt",
    method="lora",
)

# Chat format: {"messages": [{"role": "user", "content": "..."}, ...]}
state = trainlib.finetune(
    model="meta-llama/Llama-3.1-8B",
    dataset="chat_data.jsonl",
    format="chat",
    method="lora",
)

# Text format: {"text": "..."}
state = trainlib.finetune(
    model="meta-llama/Llama-3.1-8B",
    dataset="text_data.jsonl",
    format="text",
    method="lora",
)

## Data Formats

trainlib supports multiple built-in dataset formats:

### Alpaca Format
Instruction-input-output triples for supervised fine-tuning:
```json
{"instruction": "Summarize this article", "input": "...", "output": "..."}
```

### ShareGPT Format
Multi-turn conversations:
```json
{"conversations": [
    {"from": "human", "value": "Hello!"},
    {"from": "gpt", "value": "Hi there!"}
]}
```

### Chat Format
OpenAI-style chat messages:
```json
{"messages": [
    {"role": "user", "content": "Hello!"},
    {"role": "assistant", "content": "Hi there!"}
]}
```

### Text Format
Raw text for causal language modeling:
```json
{"text": "This is a raw text document..."}
```

In [ ]:
from trainlib.config.schema import DataConfig, LoraConfig, ModelConfig, TrainConfig, TrainerConfig

config = TrainConfig(
    recipe="finetune",
    method="lora",
    model=ModelConfig(name="meta-llama/Llama-3.1-8B"),
    data=DataConfig(path="data/train.jsonl", format="alpaca", eval_split=0.05),
    lora=LoraConfig(
        rank=32,
        alpha=64,
        dropout=0.1,
        target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    ),
    trainer=TrainerConfig(
        batch_size=4,
        gradient_accumulation=8,
        learning_rate=2e-4,
        num_epochs=3,
    ),
)

state = trainlib.finetune(config=config)

## Customizing LoRA Parameters

Fine-tune LoRA behavior with configuration objects:

- **rank**: Adapter matrix rank (higher = more capacity, more memory). Typical: 8-64
- **alpha**: Scaling factor, usually 2x rank. Controls adapter strength
- **dropout**: Regularization to prevent overfitting. Typical: 0.05-0.1
- **target_modules**: Which layers to adapt. Common choices:
  - Query/Value projections: `["q_proj", "v_proj"]` (minimal, fast)
  - All attention: `["q_proj", "k_proj", "v_proj", "o_proj"]` (recommended)
  - Attention + FFN: `["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]` (maximum capacity)

In [ ]:
state = trainlib.finetune(
    model="meta-llama/Llama-3.1-8B",
    dataset="data/train.jsonl",
    method="full",
    format="alpaca",
    num_epochs=3,
    learning_rate=5e-5,
    batch_size=2,
)

## Full Fine-Tuning

Full fine-tuning updates all model parameters, offering maximum flexibility and quality:

- Best results for domain-specific tasks
- Complete model adaptation
- Highest VRAM requirements
- Produces a standalone fine-tuned model

Use when you have sufficient compute resources and need the highest quality adaptation.

In [ ]:
state = trainlib.finetune(
    model="meta-llama/Llama-3.1-70B",
    dataset="data/train.jsonl",
    method="qlora",
    format="alpaca",
    num_epochs=3,
    learning_rate=2e-4,
    batch_size=4,
)

## QLoRA Fine-Tuning

QLoRA combines 4-bit quantization with LoRA adapters, enabling fine-tuning of massive models on consumer GPUs:

- Fine-tune 70B models on a single 24GB GPU
- Fine-tune 13B models on 16GB GPUs
- Minimal quality loss compared to full precision LoRA
- Same adapter portability as standard LoRA

Ideal for working with large models on limited hardware.

In [ ]:
import trainlib

# Simple LoRA fine-tuning
state = trainlib.finetune(
    model="meta-llama/Llama-3.1-8B",
    dataset="data/train.jsonl",
    method="lora",
    format="alpaca",
    num_epochs=3,
    learning_rate=2e-4,
    batch_size=4,
)

## LoRA Fine-Tuning

LoRA (Low-Rank Adaptation) trains small adapter matrices instead of updating all model parameters. This approach:

- Uses 10-100x less memory than full fine-tuning
- Trains 2-3x faster
- Produces small adapter files (typically <100MB) that can be shared separately
- Maintains base model quality while adapting to your task

Perfect for most fine-tuning tasks on consumer hardware.

# Fine-Tuning with trainlib

Fine-tuning adapts pre-trained models to your specific tasks and datasets. trainlib supports three fine-tuning methods:

- **LoRA**: Low-Rank Adaptation — memory-efficient training using small adapter matrices
- **QLoRA**: Quantized LoRA — combines 4-bit quantization with LoRA to fit large models on single GPUs
- **Full**: Full parameter fine-tuning — updates all model weights for maximum quality

Each method offers simple API access with full configuration control when you need it.